# Export Onnx Notebook

- Source: `src/export_onnx.py`
- 목적: 원본 파이썬 파일을 단계별로 실행/설명하기 위한 노트북 버전
- 실행 방법: 위에서 아래로 순서대로 실행


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    # 노트북이 다른 경로에서 열렸을 때 프로젝트 루트 자동 탐색
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / 'src').exists() and (parent / 'configs').exists():
            PROJECT_ROOT = parent
            break
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')


## Step 1. Setup and Imports

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
"""ONNX 내보내기(export) + onnxruntime 속도 측정.

왜 ONNX로 변환하나?
─────────────────
- PyTorch 모델은 Python + torch 라이브러리가 없으면 실행 불가
- ONNX 형식으로 변환하면 onnxruntime 만으로 CPU/GPU 어디서든 실행 가능
- 이 프로젝트에서는 Streamlit 앱이 .onnx 파일로 모델을 불러 빠르게 추론한다.
  (Roast 6.38ms/157FPS, Defect 4.74ms/211FPS 달성)

이 스크립트가 하는 일
─────────────────
1. best_<task>.pth 로드
2. torch.onnx.export로 .onnx 파일 생성
3. onnxruntime로 100회 반복 추론 후 평균 주기(ms)와 FPS 출력

사용 예시
─────────
    # 모델 재학습 후 ONNX 재생성할 때
    python -m src.export_onnx --config configs/default.yaml
    python -m src.export_onnx --config configs/defect.yaml
"""
from __future__ import annotations
import argparse
import time
from pathlib import Path
import numpy as np
import torch

from src.utils.config import load_config
from src.model import CoffeeClassifier
from src.evaluate import load_checkpoint


## Step 2. Function: main

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--ckpt", default=None)
    ap.add_argument("--out", default=None)
    args = ap.parse_args()

    cfg = load_config(args.config)
    task, classes = cfg["task"], cfg["classes"]
    img_size = cfg["data"]["img_size"]
    ckpt_path = args.ckpt or f"{cfg['paths']['ckpt_dir']}/best_{task}.pth"
    out_path = args.out or f"{cfg['paths']['ckpt_dir']}/best_{task}.onnx"

    # ONNX로 내보내기 전, PyTorch checkpoint를 먼저 동일하게 복원한다.
    _, state = load_checkpoint(ckpt_path, "cpu")
    ckpt_full = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    ckpt_cfg = ckpt_full.get("config") if isinstance(ckpt_full, dict) else None
    backbone = (ckpt_cfg or cfg)["model"]["name"]
    print(f"[i] using backbone from checkpoint: {backbone}")
    model = CoffeeClassifier(
        backbone=backbone, n_classes=len(classes), pretrained=False,
        dropout=cfg["model"]["dropout"], hidden=cfg["model"]["hidden"],
    )
    model.load_state_dict(state); model.eval()

    # dummy input은 export 시 그래프 shape를 추적하는 용도로만 사용된다.
    dummy = torch.randn(1, 3, img_size, img_size)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    torch.onnx.export(
        model, dummy, out_path,
        input_names=["input"], output_names=["logits"],
        # batch 축을 동적으로 두면 추후 1장 이상 배치 추론도 가능하다.
        dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
        opset_version=17,
    )
    print(f"[OK] exported {out_path}")

    # 속도 비교
    # export만 끝내지 않고, ONNX Runtime에서 실제로 얼마나 빨라지는지도 확인한다.
    import onnxruntime as ort
    sess = ort.InferenceSession(out_path, providers=["CPUExecutionProvider"])
    x = np.random.randn(1, 3, img_size, img_size).astype(np.float32)

    # warm-up
    # 첫 실행은 초기화 비용이 섞일 수 있으므로 몇 번 warm-up 후 측정한다.
    for _ in range(5):
        sess.run(None, {"input": x})
        with torch.no_grad():
            model(torch.from_numpy(x))

    N = 30
    t0 = time.perf_counter()
    for _ in range(N):
        sess.run(None, {"input": x})
    onnx_ms = (time.perf_counter() - t0) * 1000 / N

    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(N):
            model(torch.from_numpy(x))
    pt_ms = (time.perf_counter() - t0) * 1000 / N

    print(f"[BENCH] PyTorch CPU: {pt_ms:6.2f} ms/img  ({1000/pt_ms:5.1f} FPS)")
    print(f"[BENCH] ONNX    CPU: {onnx_ms:6.2f} ms/img  ({1000/onnx_ms:5.1f} FPS)")


## Step 3. Run Entry Point

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
if __name__ == "__main__":
    main()


## 실행 파라미터 가이드

- 이 파일은 원래 CLI 인자(argparse) 기반으로 동작합니다.
- 노트북에서는 인자 대신 아래처럼 변수 셀을 만들어 실행하세요.


In [ ]:
# 예시 파라미터 셀
CONFIG_PATH = 'configs/default.yaml'
CKPT_PATH = None
